# Two-Phase ESG Risk Screener

### Investment Banking — Banking-AI


            ## Step 0 — Notebook Header

            **Prerequisites:** Basic Python, familiarity with screening concepts, and comfort with classification metrics for imbalanced risk flags.

            **What You Will Learn:**

            - Describe how a light first-pass screener and a deeper agentic review can work together in ESG monitoring.
- Compare a phase-one screening model against a richer phase-two model for severe-risk detection.
- Adapt a two-stage screening pattern to another compliance or responsible-investment use case.

            **Estimated time:** 45 minutes

            **Collection context:** Investment-banking notebooks that show how broad surveillance can be combined with selective deep-dive analysis.


## Step 1 — Investment-Banking Problem Introduction

**Why this problem matters:** Small teams cannot manually deep-dive every company, so the challenge is screening broadly without losing serious supply-chain or governance issues.

**Operational question:** Which companies need a deeper ESG investigation, and which signals justify the escalation?

**Primary stakeholders:** responsible-investment teams, stewardship analysts, ESG research groups, and portfolio managers

**Decision supported:** Use a fast first-pass screen to narrow the universe, then apply richer signals to the flagged subset.

**Comprehension check:** Before looking at the data, write one sentence describing why a broad but noisy first pass can still be useful.

**Domain framing note:** This notebook is educational and synthetic. It demonstrates decision support for investment-banking and adjacent institutional workflows, not production approval or investment advice.


## Step 2 — Environment Setup

Use lightweight classification models and synthetic company-level signals to keep the full two-phase workflow transparent.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, recall_score
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)
RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
N_COMPANIES = 2000

print("Environment ready for a synthetic ESG screening workflow.")


## Step 3 — Data Creation & Context

We simulate company-level ESG signals spanning labor alerts, deforestation, supply-chain opacity, and financing-chain risk so the notebook mirrors a broad plus deep-dive screen.


In [ ]:
esg_df = pd.DataFrame({
    "controversy_count": RNG.poisson(2.2, N_COMPANIES),
    "labor_alert_count": RNG.poisson(1.4, N_COMPANIES),
    "deforestation_signal": RNG.uniform(0.0, 1.0, N_COMPANIES),
    "supply_chain_opacity": RNG.uniform(0.0, 1.0, N_COMPANIES),
    "jurisdiction_risk": RNG.uniform(0.0, 1.0, N_COMPANIES),
    "governance_red_flags": RNG.poisson(1.2, N_COMPANIES),
    "supplier_network_risk": RNG.uniform(0.0, 1.0, N_COMPANIES),
    "financing_chain_risk": RNG.uniform(0.0, 1.0, N_COMPANIES),
    "multilingual_media_risk": RNG.uniform(0.0, 1.0, N_COMPANIES),
})

phase_signal = (
    0.22 * esg_df["controversy_count"]
    + 0.26 * esg_df["labor_alert_count"]
    + 0.8 * esg_df["deforestation_signal"]
    + 0.65 * esg_df["supply_chain_opacity"]
    + 0.45 * esg_df["jurisdiction_risk"]
    + 0.3 * esg_df["governance_red_flags"]
    + 0.8 * esg_df["supplier_network_risk"]
    + 0.7 * esg_df["financing_chain_risk"]
    + 0.7 * esg_df["multilingual_media_risk"]
    + RNG.normal(0, 0.35, N_COMPANIES)
)
threshold = np.quantile(phase_signal, 0.78)
esg_df["severe_risk_flag"] = np.where(phase_signal >= threshold, "Flag", "Clear")

print(esg_df.head(3).round(3).to_string(index=False))


## Step 4 — Exploratory Data Analysis

Inspect the severe-risk balance and how controversy counts interact with environmental and supply-chain signals before modeling.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=esg_df, x="severe_risk_flag", order=["Clear", "Flag"], color="#2A9D8F", ax=axes[0])
axes[0].set_title("Severe ESG Risk Flags")
axes[0].set_xlabel("Screening Outcome")
axes[0].set_ylabel("Company Count")

sns.scatterplot(
    data=esg_df,
    x="deforestation_signal",
    y="supply_chain_opacity",
    hue="severe_risk_flag",
    alpha=0.7,
    ax=axes[1],
)
axes[1].set_title("Environmental Signal vs. Supply-Chain Opacity")
axes[1].set_xlabel("Deforestation Signal")
axes[1].set_ylabel("Supply-Chain Opacity")

plt.tight_layout()
plt.show()
plt.close()


Alt-Text: The EDA shows how environmental and supply-chain concerns cluster in the flagged set, reinforcing the rationale for a targeted second-phase review.


## Step 5 — Feature Engineering

We separate the phase-one features from the deeper phase-two features to keep the two-stage screening logic visible.


In [ ]:
analysis_df = esg_df.copy()
phase_one_features = [
    "controversy_count",
    "labor_alert_count",
    "deforestation_signal",
    "supply_chain_opacity",
    "jurisdiction_risk",
    "governance_red_flags",
]
phase_two_features = phase_one_features + [
    "supplier_network_risk",
    "financing_chain_risk",
    "multilingual_media_risk",
]

print(analysis_df[phase_two_features].head(3).round(3).to_string(index=False))


## Step 6 — Baseline Establishment

Phase 1 uses a lighter model with broad coverage; its job is to catch likely issues, not to provide the final judgment.


In [ ]:
X_train_1, X_test_1, y_train, y_test = train_test_split(
    analysis_df[phase_one_features],
    analysis_df["severe_risk_flag"],
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=analysis_df["severe_risk_flag"],
)

phase_one_model = LogisticRegression(max_iter=1200)
phase_one_model.fit(X_train_1, y_train)
phase_one_pred = phase_one_model.predict(X_test_1)

print(f"Phase 1 accuracy: {accuracy_score(y_test, phase_one_pred):.3f}")
print(f"Phase 1 recall: {recall_score(y_test, phase_one_pred, pos_label='Flag'):.3f}")


## Step 7 — Model Training & Evaluation

Phase 2 brings in deeper agent signals for the flagged set. Compare how much recall and overall fit improve.


In [ ]:
X_train_2, X_test_2, _, _ = train_test_split(
    analysis_df[phase_two_features],
    analysis_df["severe_risk_flag"],
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=analysis_df["severe_risk_flag"],
)

phase_two_model = LogisticRegression(max_iter=1200)
phase_two_model.fit(X_train_2, y_train)
phase_two_pred = phase_two_model.predict(X_test_2)

print(f"Phase 2 accuracy: {accuracy_score(y_test, phase_two_pred):.3f}")
print(f"Phase 2 recall: {recall_score(y_test, phase_two_pred, pos_label='Flag'):.3f}")
print(classification_report(y_test, phase_two_pred))


## Step 8 — Interpretability & Explainability

Explainability is crucial because a stewardship or responsible-investment analyst needs to know which risk family actually drove the escalation.


In [ ]:
importance_frame = pd.DataFrame(
    {
        "feature": phase_two_features,
        "importance": np.abs(phase_two_model.coef_[0]),
    }
).sort_values("importance", ascending=False)

sns.barplot(data=importance_frame, y="feature", x="importance", color="#6BAF46")
plt.title("Drivers of Phase 2 ESG Escalation")
plt.xlabel("Model Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()
plt.close()

print(importance_frame.head(6).round(3).to_string(index=False))


## Step 9 — Operational Application

Operationally, the workflow should show which companies are flagged in each phase and what additional context phase two contributed.


In [ ]:
sample_companies = analysis_df.sample(8, random_state=RANDOM_SEED).copy()
sample_companies["phase_one_score"] = phase_one_model.predict_proba(sample_companies[phase_one_features])[:, 1]
sample_companies["phase_one_flag"] = np.where(sample_companies["phase_one_score"] >= 0.45, "Review", "Clear")
sample_companies["phase_two_flag"] = phase_two_model.predict(sample_companies[phase_two_features])

output_columns = [
    "controversy_count",
    "labor_alert_count",
    "deforestation_signal",
    "supplier_network_risk",
    "multilingual_media_risk",
    "phase_one_score",
    "phase_one_flag",
    "phase_two_flag",
]
print(sample_companies[output_columns].round(3).to_string(index=False))


            ## Step 10 — Summary & Reflection

            **What you accomplished:** You built a synthetic two-stage ESG screening workflow that combines broad coverage with deeper, more targeted review for flagged companies.

            **Limitations to note:**

            - The data is synthetic and collapses many real-world ESG complexities into a small set of proxy signals.
- Phase boundaries are simplified; real teams may use multiple human review steps and multilingual retrieval systems.
- The output is an educational screening signal, not a complete stewardship or investment decision.

            **Ethical reflection:** A screening model can extend coverage, but it must avoid turning incomplete signals into false certainty about communities, suppliers, or labor practices.

            **Reflection questions:**

            1. What would justify moving a company to phase two even if the light model score is low?
2. Which ESG issue type is least likely to be captured by this synthetic feature set?
3. How would you build a handover path for ambiguous cases that remain unresolved after phase two?


            ## References

            - Scikit-learn user guide: logistic regression, ensemble models, and classification metrics.
- Scenario inspiration: public descriptions of ESG screening and adverse-impact monitoring workflows.
- General responsible-investment literature on controversy screening and supply-chain due diligence.
